In [1]:
# evaluating fine tuned model on qa test set using accuracy, recall, precision, f-1 score
# side note - we can also try comparing open source llms for our classification task based on this notebook https://colab.research.google.com/github/timothelaborie/text_classification_scripts/blob/main/unsloth_classification.ipynb
# requirements: xrobert model files, test dataset

import os
import io
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report
from transformers import Trainer, TrainingArguments

c:\Users\G34371231\AppData\Local\anaconda3\envs\dtais_demo_conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Evaluation for 50 random samples taken from Memo_Dataset.csv

In [2]:
path = os.getcwd() #path should be Evaluatio folder
parent = os.path.join(path, os.pardir) #parent directory
memo_dataset_path = os.path.join(parent, "Data/Memo_Dataset.csv")
df = pd.read_csv(memo_dataset_path)
print(df.head)
df = df.rename(columns={'Question_eng': 'text', 'Trigger': 'label'})
df['label'] = df['label'].astype(int)
df["orig_row_index"] = df.index

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.remove_columns(["__index_level_0__"])
val_dataset = Dataset.from_pandas(val_df)
val_dataset = val_dataset.remove_columns(["__index_level_0__"])

<bound method NDFrame.head of                                                 Question  \
0      استعملت قطرة لتوسيع حدقة العين وتوسعت، وآلان ل...   
1      أبلغ من العمر ٢٧ سنة تؤلمني رجلي عند ممارسة ال...   
2      السلام عليكمانااسمي عبدالله .وقد قمت بعمل جراح...   
3      قبل عشرة سنوات صابني كهرباء زائدة في الدماغ ور...   
4      السلام عليكم\r\nدكتور منذ شهر تقريبا بدأت اوجا...   
...                                                  ...   
13137  انا متزوج من خمس سنوات ولم يتم الانجاب بل فى و...   
13138  ماذا افعل ابنى عتده ٥ سنوات وينزل دم فى نهاية ...   
13139  هل الميكروب الحلزوني في المعده يؤثر علي صحه ال...   
13140  أنا رجل عمري (59) عاما أعاني من عدم انتصاب الق...   
13141  اريد منكم حلا و اجابتي ان كان هناك حل او علاج ...   

                                            Question_eng  Trigger  
0      I used a drop to dilate the pupil and it dilat...        1  
1      I am 27 years old. My legs hurt when I exercis...        1  
2      Peace be upon you. My name is Abdullah

In [3]:
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
import torch

In [4]:
#references: https://huggingface.co/FacebookAI/xlm-roberta-base, https://huggingface.co/docs/transformers/en/model_doc/xlm-roberta#transformers.XLMRobertaForSequenceClassification
#tokenizer and model with random head
base_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
base_model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=2)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification #we should try fine tuning XLMRobertaForSequenceClassification as well

tokenizer_ft = AutoTokenizer.from_pretrained("Tanvi0212/xlmr-trigger-detection")
unbalanced_roberta_ft_model = AutoModelForSequenceClassification.from_pretrained("Tanvi0212/xlmr-trigger-detection")

In [6]:
def test_statement(text, tokenizer, model):

    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
        pred_label_id = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_label_id].item()

    # check model’s label mapping
    print(model.config.id2label)  # should tell you {0: 'Non-trigger', 1: 'Trigger'} or similar

    pred_label = model.config.id2label[pred_label_id]

    print(f"Predicted label: {pred_label}, Confidence: {confidence:.3f}")

    return pred_label


In [7]:
test_text = "I’m bleeding through the bandage even after changing it twice today"
test_statement(text = test_text, tokenizer = tokenizer_ft, model = unbalanced_roberta_ft_model)

{0: 'LABEL_0', 1: 'LABEL_1'}
Predicted label: LABEL_1, Confidence: 0.827


'LABEL_1'

In [ ]:
def preprocess(batch, tokenizer):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    recall = recall_score(labels, preds)
    precision = precision_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "recall" : recall, "precision": precision, "f1": f1}

def run_model(dataset, model, tokenizer):
    val_tokenized = dataset.map(lambda batch: preprocess(batch, tokenizer), batched=True)
    val_tokenized = val_tokenized.rename_column("label", "labels")
    val_tokenized.set_format("torch")

    training_args = TrainingArguments(
        output_dir="./eval_results",
        per_device_eval_batch_size=8,
        do_eval=True,
        dataloader_drop_last=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        eval_dataset=val_tokenized,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    results = trainer.evaluate()
    print("Evaluation metrics:", results)
    return trainer, val_tokenized

def sample_balanced_pairs(val_df, seed=47, num_pairs=50):
    n_per_label = num_pairs // 2
    sampled_0 = val_df[val_df["label"] == 0].sample(n=n_per_label, random_state=seed)
    sampled_1 = val_df[val_df["label"] == 1].sample(n=n_per_label, random_state=seed)
    sampled = pd.concat([sampled_0, sampled_1]).sample(frac=1, random_state=seed).reset_index(drop=True)
    return sampled

def interleave_english_arabic(df_pairs):
    #formatting to copy to results spreadsheet. prompting model in the order english example, then translated arabic statement
    combined_rows = []
    for _, row in df_pairs.iterrows():
        combined_rows.append({
            "text": row["text"],
            "label": row["label"],
            "orig_row_index": row["orig_row_index"],
            "lang": "english"
        })
        combined_rows.append({
            "text": row["Question"],
            "label": row["label"],
            "orig_row_index": row["orig_row_index"],
            "lang": "arabic"
        })
    return pd.DataFrame(combined_rows)

sampled_pairs = sample_balanced_pairs(val_df, seed=47, num_pairs=50) #sampled 50 examples from test set
df_samples = interleave_english_arabic(sampled_pairs)
hf_dataset = Dataset.from_pandas(df_samples) #not necessary for only 100 examples but will use dataset library for entire test set

trainer, tokenized = run_model(hf_dataset, model = unbalanced_roberta_ft_model, tokenizer = tokenizer_ft)
preds = trainer.predict(tokenized)

y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

results_df = pd.DataFrame({
    "language": df_samples["lang"],
    "input_text": df_samples["text"],
    "orig_row_index": df_samples["orig_row_index"],
    "true_label": y_true,
    "predicted_label": y_pred
})

print("Classification Report (all examples):")
print(classification_report(y_true, y_pred, target_names=["non_trigger", "trigger"]))

print("\nSample of predictions:")
print(results_df.head(10))

results_df.to_clipboard(index=False, header=True)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map: 100%|██████████| 100/100 [00:00<00:00, 762.21 examples/s]


NameError: name 'TrainingArguments' is not defined